# Constraint programming with Choco-solver

[Choco-solver](https://choco-solver.org) is an open-source Java library for
[constraint programming](https://en.wikipedia.org/wiki/Constraint_programming) (CP): you declare
*variables* with finite domains, post *constraints* on them, and let the solver search for
solutions — or for the best one, given an objective.

Being a plain Java library, Choco can be used straight from almond. This notebook walks through a
few classic puzzles. See also [Conditional constraints](conditional_constraints.ipynb), adapted from
the [chocoteam/notebooks](https://github.com/chocoteam/notebooks) repository, and the
[Choco-solver documentation](https://choco-solver.org/docs/) for much more.

## Loading Choco

Choco-solver is a Java library, so its coordinates use a single `:` between the organization and
the module name. Choco-solver 6 requires Java 17 or later.

By default, a `Model` is pretty-printed with the full list of its variables and constraints. We
register a [pprint](https://github.com/com-lihaoyi/PPrint) handler to keep cell outputs short, and
will print the full description only when we want to see it.

In [1]:
import $ivy.`org.choco-solver:choco-solver:6.0.2`
import org.chocosolver.solver.Model
import org.chocosolver.solver.variables.IntVar

repl.pprinter() = repl.pprinter().copy(
  additionalHandlers = { case model: Model => pprint.Tree.Literal(s"Model(${model.getName})") }
)

import $ivy.$
import org.chocosolver.solver.Model
import org.chocosolver.solver.variables.IntVar

## SEND + MORE = MONEY

The classic [verbal arithmetic](https://en.wikipedia.org/wiki/Verbal_arithmetic) puzzle: assign a
distinct digit to each letter so that the addition holds, leading letters being non-zero.

A `Model` holds variables and constraints. Constraints are created by the model, and only taken
into account once `post()`-ed.

In [2]:
val sendMoreModel = new Model("SEND+MORE=MONEY")

val Seq(s, e, n, d, m, o, r, y) =
  "SENDMORY".map(c => sendMoreModel.intVar(c.toString, 0, 9))

s.ne(0).post()
m.ne(0).post()
sendMoreModel.allDifferent(s, e, n, d, m, o, r, y).post()

// SEND + MORE - MONEY = 0, written as a weighted sum
sendMoreModel.scalar(
  Array(s, e, n, d, m, o, r, e, m, o, n, e, y),
  Array(1000, 100, 10, 1, 1000, 100, 10, 1, -10000, -1000, -100, -10, -1),
  "=",
  0
).post()

println(sendMoreModel)


 Model[SEND+MORE=MONEY]

[ 9 vars -- 4 cstrs -- 0 lits -- 0 clauses ]
satisfaction  : undefined
== variables ==
S = {0..9}
E = {0..9}
N = {0..9}
D = {0..9}
M = {0..9}
O = {0..9}
R = {0..9}
Y = {0..9}
cste -- 0 = 0
== constraints ==
ARITHM ([S =/= 0])
ARITHM ([M =/= 0])
ALLDIFFERENT ([PropAllDiffInst(S, E, N, ..., Y), PropAllDiffBC(S, E, N, ..., Y), PropAllDiffAdaptative(S, E, N, ..., Y)])
SUM ([91.E + 1.D + 10.R + 1000.S - 900.O - 9000.M - 90.N - 1.Y = 0])



sendMoreModel: Model = Model(SEND+MORE=MONEY)
s: IntVar = S = {0..9}
e: IntVar = E = {0..9}
n: IntVar = N = {0..9}
d: IntVar = D = {0..9}
m: IntVar = M = {0..9}
o: IntVar = O = {0..9}
r: IntVar = R = {0..9}
y: IntVar = Y = {0..9}

`solve()` looks for the next solution. It returns `true` when one was found, leaving the variables
instantiated with its values:

In [3]:
assert(sendMoreModel.getSolver.solve())

def num(vars: IntVar*): Int = vars.foldLeft(0)(_ * 10 + _.getValue)

println(s"  ${num(s, e, n, d)}")
println(s"+ ${num(m, o, r, e)}")
println(s"= ${num(m, o, n, e, y)}")

  9567
+ 1085
= 10652


defined function num

Calling `solve()` again looks for another solution. There isn't any here:

In [4]:
sendMoreModel.getSolver.solve()

res4: Boolean = false

## N-queens

Place $n$ queens on an $n \times n$ chess board, so that no two queens attack each other. Queen $i$
goes in column $i$, and `queens(i)` is its row. Rows must all be different, and so must both kinds of
diagonals. The latter are expressed with *views*: `offset(x, k)` behaves like a variable equal to
`x + k`, without adding a variable nor a constraint to the model.

In [5]:
def queensModel(n: Int): (Model, Array[IntVar]) = {
  val model  = new Model(s"$n-queens")
  val queens = model.intVarArray("Q", n, 0, n - 1)
  model.allDifferent(queens: _*).post()
  model.allDifferent(queens.indices.map(i => model.offset(queens(i), i)): _*).post()
  model.allDifferent(queens.indices.map(i => model.offset(queens(i), -i)): _*).post()
  (model, queens)
}

def showBoard(queens: Array[IntVar]): String =
  queens.indices
    .map(row => queens.map(q => if (q.getValue == row) "♛" else "·").mkString(" "))
    .mkString("\n")

defined function queensModel
defined function showBoard

In [6]:
val (eightQueens, queens) = queensModel(8)
eightQueens.getSolver.solve()
println(showBoard(queens))

♛ · · · · · · ·
· · · · · · ♛ ·
· · · ♛ · · · ·
· · · · · ♛ · ·
· · · · · · · ♛
· ♛ · · · · · ·
· · · · ♛ · · ·
· · ♛ · · · · ·


eightQueens: Model = Model(8-queens)
queens: Array[IntVar] = Array(
  Q[0] = 0,
  Q[1] = 5,
  Q[2] = 7,
  Q[3] = 2,
  Q[4] = 6,
  Q[5] = 3,
  Q[6] = 1,
  Q[7] = 4
)
res6_1: Boolean = true

`findAllSolutions()` enumerates all the solutions, and returns them as `Solution` objects (snapshots
of the variable values). There are 92 of them on a regular chess board:

In [7]:
queensModel(8)._1.getSolver.findAllSolutions().size

res7: Int = 92

There are many more solutions on larger boards. Instead of collecting them all, `streamSolutions()`
gives them as a Java `Stream`, which we only count here. The solver also keeps statistics about its
last search:

In [8]:
val (twelveQueens, _) = queensModel(12)
val count = twelveQueens.getSolver.streamSolutions().count()
println(s"$count solutions")
println(s"${twelveQueens.getSolver.getNodeCount} nodes, ${twelveQueens.getSolver.getFailCount} failures")

14200 solutions
157345 nodes, 128946 failures


twelveQueens: Model = Model(12-queens)
count: Long = 14200L

## Sudoku

Each cell of the grid is a variable with domain $[1, 9]$, the given clues are fixed values, and each
row, column and $3 \times 3$ box must hold distinct values.

In [9]:
val puzzle = Seq(
  "53..7....",
  "6..195...",
  ".98....6.",
  "8...6...3",
  "4..8.3..1",
  "7...2...6",
  ".6....28.",
  "...419..5",
  "....8..79"
)

def solveSudoku(puzzle: Seq[String]): Option[Seq[Seq[Int]]] = {
  val model = new Model("sudoku")
  val grid = Array.tabulate(9, 9) { (i, j) =>
    puzzle(i)(j) match {
      case '.' => model.intVar(s"c_${i}_$j", 1, 9)
      case c   => model.intVar(c.asDigit)
    }
  }

  for (i <- 0 until 9) {
    model.allDifferent(grid(i): _*).post()
    model.allDifferent(grid.map(_(i)): _*).post()
    val (bi, bj) = (i / 3 * 3, i % 3 * 3)
    model.allDifferent((for (di <- 0 until 3; dj <- 0 until 3) yield grid(bi + di)(bj + dj)): _*).post()
  }

  val solver = model.getSolver
  if (solver.solve()) {
    println(s"Solved with ${solver.getNodeCount} node(s), ${solver.getFailCount} failure(s)")
    Some(grid.map(_.map(_.getValue).toSeq).toSeq)
  }
  else
    None
}

def printGrid(grid: Seq[Seq[Int]]): Unit =
  for (i <- 0 until 9) {
    if (i % 3 == 0 && i > 0) println("------+-------+------")
    println(grid(i).grouped(3).map(_.mkString(" ")).mkString(" | "))
  }

puzzle: Seq[String] = List(
  "53..7....",
  "6..195...",
  ".98....6.",
  "8...6...3",
  "4..8.3..1",
  "7...2...6",
  ".6....28.",
  "...419..5",
  "....8..79"
)
defined function solveSudoku
defined function printGrid

In [10]:
printGrid(solveSudoku(puzzle).get)

Solved with 1 node(s), 0 failure(s)
5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
------+-------+------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
------+-------+------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9


Constraint propagation alone solves this one: the search tree has a single node, and the solver
never needs to guess a value.

## Optimization: Golomb ruler

A [Golomb ruler](https://en.wikipedia.org/wiki/Golomb_ruler) is a set of marks at integer positions
such that the distances between any two marks are all distinct. We look for the shortest ruler with
$m$ marks: the model gets an objective, and the solver keeps looking for better solutions until it
proves optimality. (This model is a lighter version of the one in the
[Loading_a_solution](https://github.com/chocoteam/notebooks/blob/master/Loading_a_solution.ipynb)
notebook of chocoteam/notebooks.)

In [11]:
val marksCount = 8

val golomb = new Model("golomb")
val marks  = golomb.intVarArray("mark", marksCount, 0, marksCount * marksCount)

golomb.arithm(marks(0), "=", 0).post()
for (i <- 0 until marksCount - 1)
  golomb.arithm(marks(i + 1), ">", marks(i)).post()

locally {
  val pairs = for (i <- 0 until marksCount; j <- i + 1 until marksCount) yield (i, j)
  val diffs = pairs.map { case (i, j) => golomb.intVar(s"d_${i}_$j", 1, marksCount * marksCount) }
  for (((i, j), d) <- pairs.zip(diffs))
    golomb.arithm(marks(j), "-", marks(i), "=", d).post()
  golomb.allDifferent(diffs: _*).post()
  // symmetry breaking: the first gap is smaller than the last one
  golomb.arithm(diffs.head, "<", diffs.last).post()
}

golomb.setObjective(Model.MINIMIZE, marks.last)

marksCount: Int = 8
golomb: Model = Model(golomb)
marks: Array[IntVar] = Array(
  mark[0] = {0..64},
  mark[1] = {0..64},
  mark[2] = {0..64},
  mark[3] = {0..64},
  mark[4] = {0..64},
  mark[5] = {0..64},
  mark[6] = {0..64},
  mark[7] = {0..64}
)

Each call to `solve()` now returns a strictly better solution than the previous one, until no better
solution exists:

In [12]:
while (golomb.getSolver.solve())
  println(s"length ${marks.last.getValue}: ${marks.map(_.getValue).mkString(", ")}")
println(s"Optimal: ${golomb.getSolver.isObjectiveOptimal}")

length 46: 0, 2, 3, 8, 12, 19, 33, 46
length 43: 0, 2, 3, 8, 12, 25, 36, 43
length 42: 0, 2, 3, 10, 14, 27, 36, 42
length 40: 0, 2, 3, 10, 14, 19, 34, 40
length 39: 0, 2, 5, 11, 15, 31, 32, 39
length 37: 0, 2, 5, 18, 22, 28, 29, 37
length 35: 0, 2, 5, 16, 22, 23, 31, 35
length 34: 0, 1, 4, 9, 15, 22, 32, 34
Optimal: true
